In [1]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import os
import category_encoders as ce
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from tqdm import tqdm
import optuna
from sklearn.metrics import mean_pinball_loss
import xgboost as xgb
from xgboost.callback import EarlyStopping
from catboost import CatBoostRegressor, Pool


import warnings
warnings.filterwarnings('ignore')
print("Finished")

Finished


In [3]:
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

# To get square foot price of neighbourhood without leaking in train
def retrieve_neighbours(model, X, y, k=5, exclude_0=False):
    # For leak-free retrival of distances and prices
    # exclude_0 = True excludes the closest neighbour (typically self when train)
    X = np.array(X)
    y = np.array(y)

    if exclude_0:
        distances, indices = model.kneighbors(X, n_neighbors=k+1)
    else:
        distances, indices = model.kneighbors(X, n_neighbors=k)

    preds = []
    dists = []
    
    for d, idxs in tqdm(zip(distances, indices), total=len(indices)):

        if exclude_0:
            d = d[1:]
            idxs = idxs[1:]
        pred = np.mean(y[idxs])
        dist = np.mean(d)
    
        preds.append(pred)
        dists.append(dist)
    
    return np.array(preds), np.array(dists)

def preprocess_knn_features(X_tr, X_va, y_tr, knn_features=["latitude","longitude","sale_year"], knn_params={'n_neighbors': 10}):
    # Features based on direct neighbourhood
    scaler = StandardScaler()
    X_tr_knn = scaler.fit_transform(X_tr[knn_features])
    X_va_knn = scaler.transform(X_va[knn_features])
    knn = KNeighborsRegressor(**knn_params).fit(X_tr_knn, y_tr)

    k = knn_params["n_neighbors"]
    
    price_tr, d_tr = retrieve_neighbours(knn, X_tr_knn, y_tr, k=k, exclude_0=True)
    price_va, d_va = retrieve_neighbours(knn, X_va_knn, y_tr, k=k, exclude_0=False)

    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_tr["k_dist"], X_va["k_dist"] = d_tr, d_va
    X_tr["price_knn"], X_va["price_knn"] = price_tr, price_va

    return X_tr, X_va

train = pd.read_csv('house_price/dataset.csv')
test = pd.read_csv('house_price/test.csv')
print ("Data is loaded!")

sns.set_style("whitegrid")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

def dataset_fill_null(obj):
    obj['subdivision'].fillna('Unknown', inplace=True)
    obj['sale_nbr'].fillna('Unknown', inplace=True)
    #obj.drop(columns=['sale_nbr'], inplace=True)
    obj['submarket'].fillna('Unknown', inplace=True)


dataset_fill_null(train)
dataset_fill_null(test)
print(train.shape)
print(test.shape)

# 构造原始地址字段
train_ID = train['id']
test_ID = test['id']
# Now drop the  'Id' colum since it's unnecessary for  the prediction process.
drop_cols=['id',#row_id,没有任何信息.
           'golf',#20万数据 198756都是0,基本没什么信息了.
           'view_rainier',#20万数据,198588都是0.
           'view_skyline',#20万数据,198517都是0.
           'view_lakesamm',#20万数据,198776都是0.
           'view_otherwater',#20万数据,198473都是0.
           'view_other',#20万数据,198833都是0.
          ]
train.drop(drop_cols, axis=1, inplace=True)
test_raw = test.drop(drop_cols, axis=1, inplace=True)

# Deleting outliers
train.reset_index(drop=True, inplace=True)
# We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
# train["sale_price"] = np.log1p(train["sale_price"])
y = train.sale_price.reset_index(drop=True)

def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    df['total_baths'] = df['bath_full'] + 0.75*df['bath_3qtr'] + 0.5*df['bath_half']
    df['total_value'] = df['land_val'] + df['imp_val']
    df['living_area'] = df['sqft'] + df['sqft_fbsmt']

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_date'] = pd.to_datetime(df['sale_date'])
        df['sale_year'] = df['sale_date'].dt.year
        df['sale_month'] = df['sale_date'].dt.month
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    # 找出 object 类型列（可参与类别编码）
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    
    # 手动优先考虑的高基类列（确保存在才用）
    manual_high_card = ['sale_date','join_status', 'city', 'zoning','subdivision','submarket']
    high_card_cols = [col for col in manual_high_card if col in df.columns and df[col].dtype == 'object']
    
    # 自动补充高基类列（nunique > 50）
    for col in cat_cols:
        if col not in high_card_cols:
            try:
                n_unique = df[col].nunique()
                if n_unique > 50:
                    high_card_cols.append(col)
            except Exception as e:
                print(f"[异常] {col}: {e}")
    
    # 低基类列自动判断（nunique <= 50）
    low_card_cols = [col for col in cat_cols if col not in high_card_cols]

    print("low",low_card_cols, "high",high_card_cols)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle

# 应用预处理
# 对训练集（自己做自己）

X_train_raw, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)

test_raw = test.copy()  # 或者正确读取原始测试集
X_train = X_train_raw.copy()
X_train = X_train.drop(['sale_price'], axis=1)
X_val = X_val.drop(['sale_price'], axis=1)
X_train_full = train.drop(['sale_price'], axis=1)
X_train, encoder_bundle = preprocess_and_encode(X_train, y_train)
X_val, _ = preprocess_and_encode(X_val, encoder_bundle=encoder_bundle)
test, _ = preprocess_and_encode(test, encoder_bundle=encoder_bundle)
X_train_full, _ = preprocess_and_encode(X_train_full, y)
X_train, X_val = preprocess_knn_features(X_train, X_val, y_train)
print("Finished")

Data is loaded!
sale_nbr       42182
subdivision    17550
submarket       1717
dtype: int64
(200000, 47)
(200000, 46)
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low ['sale_nbr'] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']


100%|██████████████████████████████████| 40000/40000 [00:00<00:00, 95786.61it/s]

Finished


In [4]:
# ========= 数据划分 =========
X_tr, X_val_, y_tr, y_val_ = X_train, X_val, y_train, y_val

In [ ]:
def objective(trial):
    params = {
        "objective": "reg:quantileerror",
        "tree_method": "hist",
        "quantile_alpha": np.array([0.05, 0.95]),  # 你可以自由调整 quantiles
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }

    dtrain = xgb.QuantileDMatrix(X_train, y_train)
    dval = xgb.QuantileDMatrix(X_val, y_val, ref=dtrain)

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        early_stopping_rounds=50,
        evals=[(dval, "validation")],
        verbose_eval=False,
    )
    
    dval = xgb.DMatrix(X_val)
    preds = booster.predict(dval)
    y_lower, y_upper = preds[:, 0], preds[:, 1]

    # Winkler score（或使用其他 metric）
    score = winkler_score(y_val, y_lower, y_upper, alpha=0.1)
    return score

# 开始搜索
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

# 最优结果
print("Best trial:")
print(study.best_trial.params)

In [ ]:
# 多分位训练函数
def train_xgb_quantile(X_train, y_train, X_val, y_val, alpha_list, params):
    dtrain = xgb.QuantileDMatrix(X_train, y_train)
    dval = xgb.QuantileDMatrix(X_val, y_val, ref=dtrain)

    booster = xgb.train(
        {
            **params,
            "objective": "reg:quantileerror",
            "quantile_alpha": np.array(alpha_list),
            "tree_method": "hist",
        },
        dtrain,
        num_boost_round=params.get("n_estimators", 1000),
        early_stopping_rounds=50,
        evals=[(dval, "validation")],
        verbose_eval=False,
    )
    return booster

# 多分位预测
def predict_xgb_quantile(booster, X, ref_X=None):
    dtest = xgb.QuantileDMatrix(X, ref=xgb.QuantileDMatrix(ref_X) if ref_X is not None else None)
    return booster.inplace_predict(dtest, validate_features=False)

# 一键训练+评估函数
def evaluate_winkler(X_train, y_train, X_val, y_val, alpha=0.1, model_params=None):
    if model_params is None:
        model_params = {
            "learning_rate": 0.05,
            "max_depth": 6,
            "n_estimators": 1000,
        }

    alpha_list = [alpha / 2, 1 - alpha / 2]
    booster = train_xgb_quantile(X_train, y_train, X_val, y_val, alpha_list, model_params)
    preds = predict_xgb_quantile(booster, X_val, ref_X=X_train)
    y_lower, y_upper = preds[:, 0], preds[:, 1]
    score = winkler_score(y_val, y_lower, y_upper, alpha=alpha, return_coverage=True)
    return score, booster


In [ ]:
def train_quantile_model_lgb(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y,
              eval_set=[(X_val_, y_val_)],
               callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
                )
    return model

def objective(trial):
    common_params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 5, 64),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100, step=10),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
    }

    lower_model = train_quantile_model_lgb(X_tr, y_tr, alpha=0.05, params=common_params)
    upper_model = train_quantile_model_lgb(X_tr, y_tr, alpha=0.95, params=common_params)

    pred_lower = lower_model.predict(X_val_)
    pred_upper = upper_model.predict(X_val_)

    score = winkler_score(y_val_, pred_lower, pred_upper, alpha=0.1)
    return score  # 越小越好

# ========= 启动搜索 =========
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, n_jobs=4)
print("Best params:", study.best_params)


In [45]:
def best_features(model, num=50):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(num))



In [ ]:
def train_cb_quantile_model(X, y, alpha, params):
    params = params.copy()
    params.update({
        'loss_function': f'Quantile:alpha={alpha}',
        'verbose': 0
    })

    train_pool = Pool(X, y)
    val_pool = Pool(X_val_, y_val_)

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

    return model

def objective_cb(trial):
    common_params = {
        'iterations': trial.suggest_int('iterations', 500, 3000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }

    lower_model = train_cb_quantile_model(X_tr, y_tr, alpha=0.05, params=common_params)
    upper_model = train_cb_quantile_model(X_tr, y_tr, alpha=0.95, params=common_params)

    pred_lower = lower_model.predict(X_val_)
    pred_upper = upper_model.predict(X_val_)

    score = winkler_score(y_val_, pred_lower, pred_upper, alpha=0.1)
    return score

# 启动搜索
study = optuna.create_study(direction='minimize')
study.optimize(objective_cb, n_trials=100, n_jobs=4)

print("Best params:", study.best_params)


In [ ]:
from catboost import CatBoostRegressor, Pool

def train_cb_quantile_model(X, y, alpha, params):
    params = params.copy()
    params.update({
        'loss_function': f'Quantile:alpha={alpha}',
        'verbose': 0
    })

    train_pool = Pool(X, y)
    val_pool = Pool(X_val_, y_val_)

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

    return model
    
best_params = {'iterations': 3000, 'learning_rate': 0.04809304932404509, 'depth': 5, 'l2_leaf_reg': 7.430597979768241, 'bagging_temperature': 0.8625782660458902, 'random_strength': 6.857105332279787e-05, 'border_count': 119}

lower_model = train_cb_quantile_model(X_train, y_train, alpha=0.05, params=best_params)
upper_model = train_cb_quantile_model(X_train, y_train, alpha=0.95, params=best_params)

pred_lower = lower_model.predict(X_val)
pred_upper = upper_model.predict(X_val)

score = winkler_score(y_val, pred_lower-9000, pred_upper+9000, alpha=0.1)
print(score)


In [21]:
#  训练模型
def train_quantile_model(alpha,mdoel_name):
    if mdoel_name=='cat':
        cat_params = {
            'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
            'learning_rate': 0.05,           # 学习率
            'iterations': 8000,              # 树的数量
            'random_seed': 42,               # 随机种子
            'verbose': 800,                   # 显示训练过程
            'grow_policy' :"Depthwise",
            'min_data_in_leaf': 1000,
            'l2_leaf_reg': 100,
            'od_type':"IncToDec",
            'od_pval':0.1,
        }
        model = CatBoostRegressor(**cat_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
        )
        return model

cat_model_lower = train_quantile_model(0.05,"cat")
print("cat_model_lower is ok")
cat_model_upper = train_quantile_model(0.95,"cat")
print('cat_model_upper is ok')

cat_lower = cat_model_lower.predict(X_val)
cat_upper = cat_model_upper.predict(X_val)

score = winkler_score(y_val, cat_lower, cat_upper, alpha=0.1)
print(score)

0:	learn: 21174.0855061	test: 21125.6647813	best: 21125.6647813 (0)	total: 24.4ms	remaining: 3m 15s
800:	learn: 6944.9627326	test: 7556.4634763	best: 7556.4634763 (800)	total: 23.1s	remaining: 3m 28s
1600:	learn: 6515.0463082	test: 7390.0856143	best: 7390.0752694 (1599)	total: 45.7s	remaining: 3m 2s
2400:	learn: 6317.6151246	test: 7344.6000520	best: 7344.3830467 (2398)	total: 1m 8s	remaining: 2m 39s
3200:	learn: 6184.4063474	test: 7328.7928826	best: 7328.7928826 (3200)	total: 1m 30s	remaining: 2m 15s
4000:	learn: 6086.9772060	test: 7320.3784368	best: 7320.2897702 (3978)	total: 1m 51s	remaining: 1m 51s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 7319.132149
bestIteration = 4121

Shrink model to first 4122 iterations.
cat_model_lower is ok
0:	learn: 62620.7560805	test: 62755.0781398	best: 62755.0781398 (0)	total: 23.4ms	remaining: 3m 7s
800:	learn: 8126.8469905	test: 9374.4321567	best: 9374.4321567 (800)	total: 22.8s	remaining: 3m 25s
1600:	learn: 7509.6897416	test:

In [22]:
score = winkler_score(y_val, cat_lower-8400, cat_upper+8400, alpha=0.1, return_coverage=True)
print(score)

(324494.676749847, 0.9009)


In [23]:
best_parmas = {'n_estimators': 3000, 'learning_rate': 0.04636037450615781, 'num_leaves': 19, 'min_data_in_leaf': 100, 'feature_fraction': 0.7298036912145577, 'bagging_fraction': 0.8129812092810939, 'bagging_freq': 4, 'lambda_l1': 1.173467234602045, 'lambda_l2': 2.576496765290143}
lower_model_lgb = train_quantile_model_lgb(X_train, y_train, alpha=0.05, params=best_parmas)
upper_model_lgb = train_quantile_model_lgb(X_train, y_train, alpha=0.95, params=best_parmas)

lgb_lower = lower_model_lgb.predict(X_val)
lgb_upper = upper_model_lgb.predict(X_val)

score = winkler_score(y_val, lgb_lower, lgb_upper, alpha=0.1, return_coverage=True)
print(score)
score = winkler_score(y_val, lgb_lower-8500, lgb_upper+8500, alpha=0.1, return_coverage=True)
print(score)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2569]	valid_0's quantile: 7424.85
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's quantile: 9073.45
(329965.97540820163, 0.85805)
(326669.28165217733, 0.900275)


In [36]:
best_trial = {"n_estimators": 916, 'learning_rate': 0.12548809730659324, 'max_depth': 5, 'min_child_weight': 1.9088649367187107, 'subsample': 0.7093929284277769, 'colsample_bytree': 0.8811496368451498, 'lambda': 1.6515135857379797, 'alpha': 5.2015131670668655}

alpha = 0.1
alpha_list = [alpha / 2, 1 - alpha / 2]

booster = train_xgb_quantile(
    X_train, y_train, X_val, y_val,
    alpha_list=alpha_list,
    params=best_trial
)
dval = xgb.DMatrix(X_val)
preds = booster.predict(dval)

xgb_lower, xgb_upper = preds[:, 0], preds[:, 1]

score = winkler_score(y_val, xgb_lower-9000, xgb_upper+9000, alpha=0.1, return_coverage=True)
print(score)


(337873.78, 0.855075)


In [47]:
from scipy.optimize import minimize

lower_stack = np.vstack([lgb_lower-8500, cat_lower-8400, xgb_lower-9000]).T
upper_stack = np.vstack([lgb_upper+8500, cat_upper+8400, xgb_upper+9000]).T

def fusion_winkler_loss(weights):
    w1, w2, w3 = weights
    if np.any(np.array(weights) < 0):  # 不允许负权重
        return np.inf
    if not np.isclose(w1 + w2 + w3, 1.0):
        return np.inf
    pred_lower = lower_stack @ np.array([w1, w2, w3])
    pred_upper = upper_stack @ np.array([w1, w2, w3])
    score, coverage = winkler_score(y_val, pred_lower, pred_upper, alpha=0.1, return_coverage=True)
    return score

# 初始权重
x0 = np.array([1/3, 1/3, 1/3])

# 约束: 和为1
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
# 约束: 每个 >= 0
bounds = [(0, 1)] * 3

res = minimize(fusion_winkler_loss, x0, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x
print("Best weights:", best_weights)
print("Best winkler score:", res.fun)

# 应用融合
final_lower = lower_stack @ best_weights
final_upper = upper_stack @ best_weights

Best weights: [0.29464023 0.49786104 0.20749873]
Best winkler score: 322226.0355290543


In [ ]:
#  训练模型
def train_quantile_model_cat_full(alpha):
    cat_params = {
        'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
        'learning_rate': 0.05,           # 学习率
        'iterations': 8000,              # 树的数量
        'random_seed': 42,               # 随机种子
        'verbose': 800,                   # 显示训练过程
        'grow_policy' :"Depthwise",
        'min_data_in_leaf': 1000,
        'l2_leaf_reg': 100,
        'od_type':"IncToDec",
        'od_pval':0.1,
    }
    model = CatBoostRegressor(**cat_params)
    model.fit(X_train_full, y)
    return model

cat_model_lower_full = train_quantile_model_cat_full(0.05)
cat_model_upper_full = train_quantile_model_cat_full(0.95)

cat_lower = cat_model_lower_full.predict(test)
cat_upper = cat_model_upper_full.predict(test)
print("Finished")

0:	learn: 21189.2367502	total: 38.7ms	remaining: 5m 9s
800:	learn: 7057.9158916	total: 28.6s	remaining: 4m 16s
1600:	learn: 6683.9040820	total: 55s	remaining: 3m 39s
2400:	learn: 6478.4383065	total: 1m 23s	remaining: 3m 13s
3200:	learn: 6356.0029374	total: 1m 51s	remaining: 2m 46s
4000:	learn: 6267.2754357	total: 2m 19s	remaining: 2m 19s
4800:	learn: 6196.7022900	total: 2m 46s	remaining: 1m 50s
5600:	learn: 6135.9011413	total: 3m 12s	remaining: 1m 22s
6400:	learn: 6087.9216930	total: 3m 39s	remaining: 54.9s
7200:	learn: 6048.9173157	total: 4m 6s	remaining: 27.3s
7999:	learn: 6004.8716314	total: 4m 33s	remaining: 0us
0:	learn: 62698.2056431	total: 28.3ms	remaining: 3m 46s
800:	learn: 8143.3422303	total: 26.7s	remaining: 3m 59s
1600:	learn: 7645.6443273	total: 53.2s	remaining: 3m 32s
2400:	learn: 7387.5971654	total: 1m 21s	remaining: 3m 9s
3200:	learn: 7229.9396693	total: 1m 50s	remaining: 2m 44s
4000:	learn: 7115.0513513	total: 2m 20s	remaining: 2m 20s
4800:	learn: 7007.6026908	total: 2

In [ ]:

best_trial = {"n_estimators": 916, 'learning_rate': 0.12548809730659324, 'max_depth': 5, 'min_child_weight': 1.9088649367187107, 'subsample': 0.7093929284277769, 'colsample_bytree': 0.8811496368451498, 'lambda': 1.6515135857379797, 'alpha': 5.2015131670668655}

alpha = 0.1
alpha_list = [alpha / 2, 1 - alpha / 2]

booster = train_xgb_quantile(X_full, y_full, None, None, alpha_list, best_trial)

dval = xgb.DMatrix(test)
preds = booster.predict(test)

xgb_lower, xgb_upper = preds[:, 0], preds[:, 1]
print("Finished")

In [ ]:
def train_quantile_model_lgb_full(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model
    
#best_params = {'n_estimators': 3000, 'learning_rate': 0.017548411396214044, 'num_leaves': 54, 'min_data_in_leaf': 100, 'feature_fraction': 0.6454574945988764, 'bagging_fraction': 0.8467545991300843, 'bagging_freq': 3, 'lambda_l1': 4.291976770761236, 'lambda_l2': 3.428080554361697}
best_parmas = {'n_estimators': 3000, 'learning_rate': 0.04636037450615781, 'num_leaves': 19, 'min_data_in_leaf': 100, 'feature_fraction': 0.7298036912145577, 'bagging_fraction': 0.8129812092810939, 'bagging_freq': 4, 'lambda_l1': 1.173467234602045, 'lambda_l2': 2.576496765290143}
lower_model_lgb = train_quantile_model_lgb_full(X_train_full, y, alpha=0.05, params=best_params)
upper_model_lgb = train_quantile_model_lgb_full(X_train_full, y, alpha=0.95, params=best_params)

pred_lower_lgb = lower_model_lgb.predict(test)
pred_upper_lgb = upper_model_lgb.predict(test)
print("Finished")

In [ ]:
lower_stack = np.vstack([lgb_lower-8500, cat_lower-8400, xgb_lower-9000]).T
upper_stack = np.vstack([lgb_upper+8500, cat_upper+8400, xgb_upper+9000]).T
final_lower = lower_stack @ best_weights
final_upper = upper_stack @ best_weights
print("Finished")

In [ ]:
submission = pd.DataFrame({
        "id": test_ID,
        "pi_lower": pre_low,
        "pi_upper": pre_high
    })
submission.to_csv("submission7.csv", index=False)

In [ ]:
宽度建模（残差预ce
penalized Winkler + coverage